In [3]:
import matplotlib.pyplot as plt
import numpy as np
import time
TOL = 1e-6 #tolerance for floating point comparisons
def is_close(p1, p2, tol=TOL): #says whether two coordinates are within a tolerance for error of each other
    return abs(p1[0] - p2[0]) < tol and abs(p1[1] - p2[1]) < tol

def in_path_with_tol(p, path, tol=TOL): #returns whether a point is in a path with a given tolerance for error
    return any(is_close(p, q, tol) for q in path)
def starts2(n): #generates a line of n successive points of honeycomb lattice boundary
    s=[]
    for i in range(0,n+1):
        s.append([[-0.5-i*1.5,3**0.5/2*(1-i)],[-1-i*1.5,3**0.5/2*(2-i)]])
    return s
def ends2(n, o): #generates n successive inner or outer boundary points, o==1 gives inner, o==0 for outer
    e=[]
    for i in range(0,n+1):
        if o==1:
            e.append([[0.5+i*1.5,3**0.5/2*(1-i)],[1+i*1.5,3**0.5/2*(2-i)]])
        elif o==0:
            val=2*(i+1.5)*3**0.5/2
            e.append([[0.5, val],[-0.5, val]])
    return e
def alls(length,o): #returns all points on the boundary of given length
    return(starts2(length)+ends2(length,o))
def hits(a, b): #returns whether two paths intersect
    return any(in_path_with_tol(p, b) for p in a)
def firsts(paths): #returns the first points in a set of paths
    return [p[0] for p in paths]
def lasts(paths): #returns the last points in a set of paths
    return [p[-1] for p in paths]
def generate_saws(length, start_paths, end_paths, intersect_path): #generates the walks on the boundary of length n, beginning at start_paths,
    results = [] # ending at end_paths, and intersecting intersect_path
    end_firsts = firsts(end_paths)
    start_firsts = firsts(start_paths)
    all_end_start = start_firsts + end_firsts
    def recurse(path): #recursively call this function to avoid enumerating every path
        if len(path) == length + 1:
            if in_path_with_tol(path[-1], end_firsts):
                if hits(path, intersect_path):
                    if not hits(path[1:-1], all_end_start):
                        if not any(all(is_close(p1, p2) for p1, p2 in zip(path[::-1], r)) for r in results):
                            results.append(path.copy())
            return
        last=path[-1]
        second_last=path[-2]
        dx = last[0]-second_last[0]
        dy = last[1]-second_last[1]
        left=[last[0]+dx/2-(3**0.5/2)*dy,last[1]+(3**0.5/2)*dx+dy/2]
        right=[last[0]+dx/2+(3**0.5/2)*dy,last[1]-(3**0.5/2)*dx+dy/2]
        #based on previous position generate 60-degree left and right turns
        for next_point in [left, right]:
            if not in_path_with_tol(next_point, path):
                if in_path_with_tol(last, end_firsts) and len(path)!=length:
                    continue
                    #allows the recursion to continue when the path is self-avoiding and has not hit an endpoint yet
                path.append(next_point)
                recurse(path)
                path.pop()
    for sp in start_paths:
        recurse(sp[:])  # make a copy
    return results
def generate_loops(length, start_paths, end_paths, intersect_path): #generates the walks on the boundary of length n, beginning at start_paths,
    results = [] # ending at end_paths, and intersecting intersect_path
    end_firsts = firsts(end_paths)
    start_firsts = firsts(start_paths)
    all_end_start = start_firsts + end_firsts
    def recurse(path): #recursively call this function to avoid enumerating every path
        if len(path) == length + 1:
            if in_path_with_tol(path[-1], end_firsts):
                if hits(path, intersect_path):
                    if not hits(path[1:-1], all_end_start):
                        if not any(all(is_close(p1, p2) for p1, p2 in zip(path[::-1], r)) for r in results):
                            results.append(path.copy())
            return
        last=path[-1]
        second_last=path[-2]
        dx = last[0]-second_last[0]
        dy = last[1]-second_last[1]
        left=[last[0]+dx/2-(3**0.5/2)*dy,last[1]+(3**0.5/2)*dx+dy/2]
        right=[last[0]+dx/2+(3**0.5/2)*dy,last[1]-(3**0.5/2)*dx+dy/2]
        #based on previous position generate 60-degree left and right turns
        for next_point in [left, right]:
            if not in_path_with_tol(next_point, path[1:]):
                if in_path_with_tol(last, end_firsts) and len(path)!=length:
                    continue
                    #allows the recursion to continue when the path is self-avoiding and has not hit an endpoint yet
                path.append(next_point)
                recurse(path)
                path.pop()
    for sp in start_paths:
        recurse(sp[:])  # make a copy
    return results
def sww(length_gamma,upper_bound): #generate list of numbers in s_{w=length_gamma,w<=upper_bound} 
    a=[[] for _ in range(upper_bound-2)] #initialize list
    gammas_outer=generate_saws(length_gamma,alls(upper_bound,0),alls(upper_bound,0),lasts(alls(upper_bound,1)))
    #set of gammas for outer boundary of length n
    gammas_inner=generate_saws(length_gamma,alls(upper_bound,1),alls(upper_bound,1),lasts(alls(upper_bound,0))) 
    #set of gammas for inner boundary of length n
    for self_avoiding_walk_inner in gammas_inner:
        for length_gamma_prime in range(3,upper_bound+1):
            a[length_gamma_prime-3].append(len(generate_saws(length_gamma_prime,alls(upper_bound+2,1),alls(upper_bound+2,1),self_avoiding_walk_inner))) 
            #append number of self-avoiding walks of length length_gamma_prime intersecting self_avoiding_walk_inner 
    for self_avoiding_walk_outer in gammas_outer:
        for length_gamma_prime in range(3,upper_bound+1):
            a[length_gamma_prime-3].append(len(generate_saws(length_gamma_prime,alls(upper_bound+2,0),alls(upper_bound+2,0),self_avoiding_walk_outer))) 
            #append number of self-avoiding walks of length length_gamma_prime intersecting self_avoiding_walk_outer
    swws=[0] #initialize sww
    sumlist=np.cumsum(a, axis=0) #create list of sums of each length for each path
    for i in range(len(sumlist)):
        swws.append(max(sumlist[i])) #find max values of each sum
    return(np.diff(swws)) #print the difference between successive terms
    #this function takes 1-10 minutes to run for odd length_gamma and about 1-4 hours to run for even length_gamma when upper_bound=21
def shift(paths,dx,dy): #shifts a path in the x direction by dx and y direction by dy
    result=[]
    for path in paths:
        result.append([path[0]+dx,path[1]+dy])
    return(result)
def loop_translates(m):
    toploop=[[-0.5, 2.598076211353316], [-1.0, 3.4641016151377544], [-0.5000000000000003, 4.330127018922193], 
 [0.49999999999999917, 4.330127018922193], [0.9999999999999989, 3.4641016151377544],
 [0.499999999999999, 2.598076211353316], [-0.5000000000000007, 2.598076211353316]] 
    #loop that sits above the corner#generates m translates of toploop along the line parallel to the starting points
    w=[]
    for ii in range(0,m):
        a=toploop
        w.append(shift(a,-ii*3/2,ii*(-3**.5/2)))
    return(w)
def sl6w(upper_bound): #returns list of s_{l=6,w<=upper_bound}
    a=[[] for _ in range(upper_bound-2)] #initialize list
    for loop1 in loop_translates(upper_bound): #iterate through m loops along the boundary
        for length_gamma_prime in range(3,upper_bound+1): #for each length of gamma'
            a[length_gamma_prime-3].append(len(generate_saws(length_gamma_prime,alls(upper_bound+2,0),alls(upper_bound+2,0),loop1))) 
            #append number of self-avoiding walks of length length_gamma_prime on the outer boundary intersecting loop1 
    for loop2 in loop_translates(upper_bound): #iterate through m loops along the boundary
        for length_gamma_prime in range(3,upper_bound+1): #for each length of gamma'
            a[length_gamma_prime-3].append(len(generate_saws(length_gamma_prime,alls(upper_bound+2,1),alls(upper_bound+2,1),loop2))) 
            #append number of self-avoiding walks of length length_gamma_prime on the inner boundary intersecting loop2
    slw=[0]
    sumlist=np.cumsum(a, axis=0) #create list of sums of all lengths of gamma' up to the row number
    for i in range(len(sumlist)):
        slw.append(max(sumlist[i])) #add maximums of all sums to list
    return(np.diff(slw)) #return differences between them
    #this function takes about 1-2 hours when upper_bound=21
def Qb(length,o): #generates all odd walks <=length on inner/outer boundary
    a=[0]*int((length-3)//2+1)
    for i in range(3,length+2,2):
        a[i//2-1]+=len(generate_saws(i,starts2(length),ends2(length,o),firsts(ends2(length,o))))
    return(a)
def Q(length): #returns maximum of between boundaries of odd walk counts of length<=input
    return(np.maximum(Qb(length,1),Qb(length,0)))
def saw_with_avoidance(length, start_paths, end_paths, intersect_path, avoid_path):
    #generates the walks on the boundary of length n, beginning at start_paths,
    results = [] # ending at end_paths, and intersecting intersect_path, and avoiding avoid_path
    end_firsts = firsts(end_paths)
    start_firsts = firsts(start_paths)
    def recurse(path): #recursively call this function to avoid enumerating every path
        if len(path) == length + 1:
            if in_path_with_tol(path[-1], end_firsts):
                if hits(path, intersect_path):
                    if not hits(path[1:-1], avoid_path):
                        if not any(all(is_close(p1, p2) for p1, p2 in zip(path[::-1], r)) for r in results):
                            results.append(path.copy())
            return
        last=path[-1]
        second_last=path[-2]
        dx = last[0]-second_last[0]
        dy = last[1]-second_last[1]
        left=[last[0]+dx/2-(3**0.5/2)*dy,last[1]+(3**0.5/2)*dx+dy/2]
        right=[last[0]+dx/2+(3**0.5/2)*dy,last[1]-(3**0.5/2)*dx+dy/2]
        #based on previous position generate 60-degree left and right turns
        for next_point in [left, right]:
            if not in_path_with_tol(next_point, path):
                if in_path_with_tol(last, end_firsts) and len(path)!=length:
                    continue
                    #allows the recursion to continue when the path is self-avoiding and has not hit an endpoint yet
                path.append(next_point)
                recurse(path)
                path.pop()
    for sp in start_paths:
        recurse(sp[:])  # make a copy
    return results
def R(length): #returns number of lines hitting a fixed bond in the boundary 
    # and returning to the boundary to the left of the fixed bond
    a=[]
    z=len(starts2(length))//2
    for i in range(4,length+2,2):
        a.append(len(saw_with_avoidance(i,[starts2(length)[z]],starts2(length)[:z],firsts(starts2(length)),firsts(starts2(length)[z:]))))
    return(a)
def N(length): #counts all loops of even length<=input
    a=[]
    for i in range(6,length+2,2):
        a.append(len(generate_loops(i,starts2(0),starts2(0),starts2(0)[0])))
    return(a)
def layered_loop_translates(mm,l): #generates closest mm loops from the corner and l from the boundary
    z=loop_translates(mm)
    for i in loop_translates(mm):
        for j in range(1,l+1):
            z.append(shift(i,0,3**.5*j))
    return(z)
def get_edges(list_of_paths): #returns list of all edges in a list of paths
    edges=[]
    for path in list_of_paths:
        for i in range(len(path)-1):
            edges.append([path[i],path[(i+1)%len(path)]])
    return(edges)
def get_points(list_of_paths): #returns list of all points in a list of paths
    points=[]
    for path in list_of_paths:
        for i in range(len(path)-1):
            points.append(path[i])
    return(points)
def Pb(length,o): #calculates upper bound on the number of walks<=length to the boundary (o=0/1) from any edge
    results=[]
    for i in range(1,length+1):
        running_max=[]
        for point in get_points(get_edges(layered_loop_translates(length,length))): #find all points from all edges
            a=0
            for edge in get_edges(layered_loop_translates(length,length)):
                if in_path_with_tol(point,edge) and is_close(edge[0],point):
                    a+=(len(saw_with_avoidance(i,[edge],alls(length,o),firsts(alls(length,o)),firsts(alls(length,o)))))
                    # adds to a if the point is the starting point of a walk of length i and reaches the boundary 
            running_max.append(a) #adds a after each edge is iterated through
        results.append(max(running_max)) #appends maximum over all points
    return(results)
def P(length): #calculates upper bound on the number of walks<=length to any boundary from any edge
    return(np.maximum(Pb(length,0),Pb(length,1)))
def plot2(p): #plots paths visually
    plt.plot(*np.array(p).T)
print(sww(11,21))
    

[    1     6     2     6     7    24    20    54    64   156   202   450
   646  1289  2070  3892  6613 11658 21114]


[ 1  2  2  4  6  8 16 24 40 64]
